# Fold-2 Segmentation Engine — Full Training Run

Self-contained: **code + data both ship inside `fold2_dataset.zip`** (built by
`prepare_dataset.py` + a `code/` folder), so no GitHub access is needed and the
repo can stay private. See `TASK_09_segmentation_engine.md` and
`FOLD2_DATASET_SPEC.md` for the full spec.

**Before running:** set the runtime to GPU (Runtime → Change runtime type →
T4 GPU) and upload `fold2_dataset.zip` to your Drive (`MyDrive/` root).
Fold-2 trains the full `config.yaml` recipe: 60 epochs, cosine schedule,
early stopping (patience 10), fold-2 pos_weight, offline dihedral-8 + online
scale-jitter/photometric augmentation.

In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio pyyaml


In [ ]:
# --- Setup: everything comes from the one zip in Drive ---------------------
from google.colab import drive
drive.mount('/content/drive')

DATA_ZIP = '/content/drive/MyDrive/fold2_dataset.zip'   # <-- adjust if elsewhere
!rm -rf /content/run
!unzip -q "{DATA_ZIP}" -d /content/run
# Layout inside the zip: code/ (training scripts + config) and
# fold2_dataset/ (images/masks/splits). Stage data as code/data.
%cd /content/run/code
!rm -rf data && mv /content/run/fold2_dataset data
!echo "train entries:" $(wc -l < data/splits/train.txt) "| val entries:" $(wc -l < data/splits/val.txt)

# Confirm a GPU is attached: Runtime -> Change runtime type -> T4 GPU
import torch
print("CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - fix runtime")

In [ ]:
# Sanity check: confirm data contract before burning a training run on
# a malformed dataset
import os
assert os.path.exists('data/images'), 'data/images missing'
assert os.path.exists('data/masks'), 'data/masks missing'
assert os.path.exists('data/splits/train.txt'), 'train split missing'
assert os.path.exists('data/splits/val.txt'), 'val split missing'

import numpy as np
import rasterio
from dataset import ADDON_INDEX
sample_id = open('data/splits/train.txt').readline().strip()
# images/  = addon "<stem>_satellite.png" (RGB); masks/ = "<stem>_mask_index.png"
with rasterio.open(f'data/images/{sample_id}.png') as src:
    print('image shape:', src.read().shape, 'dtype:', src.dtypes)
with rasterio.open(f'data/masks/{sample_id}.png') as src:
    idx = src.read(1)  # single-channel class-index map (multi-class, not 4-band)
    print('mask (index map) shape:', idx.shape, 'dtype:', idx.dtype)
    print('unique class indices present:', np.unique(idx))
    print('Task 09 channels <- addon index:', ADDON_INDEX)
    print('dataset.py expands this single-channel index map into 4 binary channels')

In [ ]:
# Fold-2 runs the FULL config (60 epochs, cosine, early stopping patience 10,
# fold-2 pos_weight already baked into config.yaml). For a quick smoke test
# instead, uncomment the override below.
# import yaml
# with open('config.yaml') as f: cfg = yaml.safe_load(f)
# cfg['train']['epochs'] = 10; cfg['train']['batch_size'] = 4
# with open('config_trial.yaml', 'w') as f: yaml.safe_dump(cfg, f)
print('Using full config.yaml (fold-2 recipe)')

In [ ]:
!python train.py --config config.yaml

In [ ]:
# Persist outputs — /content is wiped when the runtime ends.
!mkdir -p /content/drive/MyDrive/zoning_seg_engine_out
!cp -v checkpoints/best_model.pt /content/drive/MyDrive/zoning_seg_engine_out/best_model_fold2.pt \
    || echo "no checkpoint written (did training reach a best epoch?)"
print("Saved to MyDrive/zoning_seg_engine_out/best_model_fold2.pt")

## Reading the results
Check per-class IoU independently — do not judge the run on the mean alone.
`parcel_border` may lag the other three; before treating it as a bug, pull
up a few failing validation tiles and check whether the boundary is
actually visible in the imagery (see TASK_09 'known hard case').